In [1]:
import sys
from pathlib import Path

import pandas as pd
import geopandas as gpd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import INTERIM_DIR, PROCESSED_DIR, RAW_DIR
from src.utils import preview, fill_with_median, standardize_columns

In [2]:
nyc_features_gdf = gpd.read_file(INTERIM_DIR / "nyc_features_clean.geojson")
preview(nyc_features_gdf, "NYC Features Clean")
nyc_features_gdf.head()

NYC Features Clean: 206 rows x 9 cols
  zip_code geoid20   boroname  total_population  median_household_income  \
0    10028   10028  Manhattan             45679                 168125.0   
1    11427   11427     Queens             25124                  92129.0   
2    11697   11697     Queens              4123                 134844.0   
3    11004   11004     Queens             14296                 109865.0   
4    10282   10282  Manhattan              5960                 250001.0   

   nearest_tj_distance_ft  nearest_tj_distance_miles  subway_station_count  \
0             6500.867327                   1.231225                   2.0   
1            28540.686549                   5.405433                   0.0   
2            52999.852120                  10.037851                   0.0   
3            38817.224905                   7.351747                   0.0   
4             4415.011460                   0.836176                   0.0   

                                    

,zip_code,geoid20,boroname,total_population,median_household_income,nearest_tj_distance_ft,nearest_tj_distance_miles,subway_station_count,geometry
0,10028,10028,Manhattan,45679,168125.0,6500.867327,1.231225,2.0,"POLYGON ((994561.592 222839.437, 994690.277 22..."
1,11427,11427,Queens,25124,92129.0,28540.686549,5.405433,0.0,"POLYGON ((1049040.311 204011.201, 1049258.374 ..."
2,11697,11697,Queens,4123,134844.0,52999.852120,10.037851,0.0,"POLYGON ((1000721.481 136681.744, 1000888.65 1..."
3,11004,11004,Queens,14296,109865.0,38817.224905,7.351747,0.0,"POLYGON ((1061069.68 214228.676, 1061117.367 2..."
4,10282,10282,Manhattan,5960,250001.0,4415.011460,0.836176,0.0,"POLYGON ((979421.96 199752.189, 979617.937 201..."


In [3]:
print("shape:", nyc_features_gdf.shape)
print("duplicate zip_code:", nyc_features_gdf["zip_code"].duplicated().sum())
print("missing total_population:", nyc_features_gdf["total_population"].isna().sum())
print("missing median_household_income:", nyc_features_gdf["median_household_income"].isna().sum())
print("missing nearest_tj_distance_miles:", nyc_features_gdf["nearest_tj_distance_miles"].isna().sum())
print("missing subway_station_count:", nyc_features_gdf["subway_station_count"].isna().sum())
print("crs:", nyc_features_gdf.crs)

shape: (206, 9)
duplicate zip_code: 0
missing total_population: 0
missing median_household_income: 30
missing nearest_tj_distance_miles: 0
missing subway_station_count: 0
crs: EPSG:2263


In [4]:
nyc_features_gdf["median_household_income_filled"] = fill_with_median(
    nyc_features_gdf["median_household_income"]
)

print(
    "remaining missing after fill:",
    nyc_features_gdf["median_household_income_filled"].isna().sum()
)

print(
    "overall median used for fill:",
    nyc_features_gdf["median_household_income"].median()
)

remaining missing after fill: 0
overall median used for fill: 86543.0


In [5]:
#nyc_features_gdf["income_was_imputed"] = nyc_features_gdf["median_household_income"].isna()

In [6]:
nyc_features_gdf[[
    "zip_code",
    "boroname",
    "median_household_income",
    "median_household_income_filled"
]].head(10)

,zip_code,boroname,median_household_income,median_household_income_filled
0,10028,Manhattan,168125.0,168125.0
1,11427,Queens,92129.0,92129.0
2,11697,Queens,134844.0,134844.0
3,11004,Queens,109865.0,109865.0
4,10282,Manhattan,250001.0,250001.0
5,10027,Manhattan,64220.0,64220.0
6,11432,Queens,75395.0,75395.0
7,10452,Bronx,37854.0,37854.0
8,10469,Bronx,76018.0,76018.0
9,11225,Brooklyn,85201.0,85201.0


In [7]:
model_cols = [
    "zip_code",
    "boroname",
    "total_population",
    "median_household_income",
    "median_household_income_filled",
    "nearest_tj_distance_miles",
    "subway_station_count",
    "geometry"
]

nyc_model_ready_gdf = nyc_features_gdf[model_cols].copy()
preview(nyc_model_ready_gdf, "NYC Model Ready")
nyc_model_ready_gdf.head()

NYC Model Ready: 206 rows x 8 cols
  zip_code   boroname  total_population  median_household_income  \
0    10028  Manhattan             45679                 168125.0   
1    11427     Queens             25124                  92129.0   
2    11697     Queens              4123                 134844.0   
3    11004     Queens             14296                 109865.0   
4    10282  Manhattan              5960                 250001.0   

   median_household_income_filled  nearest_tj_distance_miles  \
0                        168125.0                   1.231225   
1                         92129.0                   5.405433   
2                        134844.0                  10.037851   
3                        109865.0                   7.351747   
4                        250001.0                   0.836176   

   subway_station_count                                           geometry  
0                   2.0  POLYGON ((994561.592 222839.437, 994690.277 22...  
1                

,zip_code,boroname,total_population,median_household_income,median_household_income_filled,nearest_tj_distance_miles,subway_station_count,geometry
0,10028,Manhattan,45679,168125.0,168125.0,1.231225,2.0,"POLYGON ((994561.592 222839.437, 994690.277 22..."
1,11427,Queens,25124,92129.0,92129.0,5.405433,0.0,"POLYGON ((1049040.311 204011.201, 1049258.374 ..."
2,11697,Queens,4123,134844.0,134844.0,10.037851,0.0,"POLYGON ((1000721.481 136681.744, 1000888.65 1..."
3,11004,Queens,14296,109865.0,109865.0,7.351747,0.0,"POLYGON ((1061069.68 214228.676, 1061117.367 2..."
4,10282,Manhattan,5960,250001.0,250001.0,0.836176,0.0,"POLYGON ((979421.96 199752.189, 979617.937 201..."


In [8]:
print("shape:", nyc_model_ready_gdf.shape)
print("duplicate zip_code:", nyc_model_ready_gdf["zip_code"].duplicated().sum())
print("missing filled income:", nyc_model_ready_gdf["median_household_income_filled"].isna().sum())
print("missing nearest_tj_distance_miles:", nyc_model_ready_gdf["nearest_tj_distance_miles"].isna().sum())
print("missing subway_station_count:", nyc_model_ready_gdf["subway_station_count"].isna().sum())
print("crs:", nyc_model_ready_gdf.crs)

shape: (206, 8)
duplicate zip_code: 0
missing filled income: 0
missing nearest_tj_distance_miles: 0
missing subway_station_count: 0
crs: EPSG:2263


In [9]:
nyc_model_ready_gdf.to_file(
    PROCESSED_DIR / "nyc_model_ready.geojson",
    driver="GeoJSON"
)

nyc_model_ready_gdf.drop(columns=["geometry"], errors="ignore").to_csv(
    PROCESSED_DIR / "nyc_model_ready.csv",
    index=False
)

In [10]:
print("Saved files:")
print(PROCESSED_DIR / "nyc_model_ready.geojson")
print(PROCESSED_DIR / "nyc_model_ready.csv")

Saved files:
/Users/mihirgaudani/trader-joes-nyc-case-study/data/processed/nyc_model_ready.geojson
/Users/mihirgaudani/trader-joes-nyc-case-study/data/processed/nyc_model_ready.csv


In [11]:
rent_df = pd.read_csv(RAW_DIR / "commercial_rent_top10.csv")
rent_df.head()

,rank,zip_code,borough,neighborhood_proxy,retail_corridor_proxy,avg_asking_rent_psf,source_name,source_period,notes
0,1,10028,Manhattan,Upper East Side,Third Avenue (60th Street - 72nd Street),259,REBNY Manhattan Retail Report,H1 2024,Used Third Avenue as Upper East Side proxy
1,2,10128,Manhattan,Upper East Side,East 86th Street (Lexington Avenue - Second Av...,297,REBNY Manhattan Retail Report,H1 2024,Used East 86th Street as closer proxy for 10128
2,3,11215,Brooklyn,Park Slope,Seventh Ave (Union St - Ninth Street),141,REBNY Brooklyn Retail Report,2024,Direct Park Slope corridor proxy
3,4,11226,Brooklyn,Flatbush / Prospect Lefferts Gardens,Flatbush Ave (Fifth Ave - Grand Army Plaza),105,REBNY Brooklyn Retail Report,2024,Used nearest Flatbush/Prospect Heights corrido...
4,5,10036,Manhattan,Midtown West / Bryant Park,Fifth Avenue (42nd Street - 49th Street),583,REBNY Manhattan Retail Report,H1 2024,Used closest Bryant Park / Midtown East-facing...


In [12]:
rent_df = standardize_columns(rent_df)
rent_df.head()

,rank,zip_code,borough,neighborhood_proxy,retail_corridor_proxy,avg_asking_rent_psf,source_name,source_period,notes
0,1,10028,Manhattan,Upper East Side,Third Avenue (60th Street - 72nd Street),259,REBNY Manhattan Retail Report,H1 2024,Used Third Avenue as Upper East Side proxy
1,2,10128,Manhattan,Upper East Side,East 86th Street (Lexington Avenue - Second Av...,297,REBNY Manhattan Retail Report,H1 2024,Used East 86th Street as closer proxy for 10128
2,3,11215,Brooklyn,Park Slope,Seventh Ave (Union St - Ninth Street),141,REBNY Brooklyn Retail Report,2024,Direct Park Slope corridor proxy
3,4,11226,Brooklyn,Flatbush / Prospect Lefferts Gardens,Flatbush Ave (Fifth Ave - Grand Army Plaza),105,REBNY Brooklyn Retail Report,2024,Used nearest Flatbush/Prospect Heights corrido...
4,5,10036,Manhattan,Midtown West / Bryant Park,Fifth Avenue (42nd Street - 49th Street),583,REBNY Manhattan Retail Report,H1 2024,Used closest Bryant Park / Midtown East-facing...


In [13]:
rent_df["zip_code"] = rent_df["zip_code"].astype(str).str.zfill(5)
rent_df["avg_asking_rent_psf"] = pd.to_numeric(rent_df["avg_asking_rent_psf"], errors="coerce")

In [14]:
print("shape:", rent_df.shape)
print("duplicate zip_code:", rent_df["zip_code"].duplicated().sum())
print("missing avg_asking_rent_psf:", rent_df["avg_asking_rent_psf"].isna().sum())

rent_df[[
    "rank",
    "zip_code",
    "borough",
    "retail_corridor_proxy",
    "avg_asking_rent_psf"
]]

shape: (10, 9)
duplicate zip_code: 0
missing avg_asking_rent_psf: 0


,rank,zip_code,borough,retail_corridor_proxy,avg_asking_rent_psf
0,1,10028,Manhattan,Third Avenue (60th Street - 72nd Street),259
1,2,10128,Manhattan,East 86th Street (Lexington Avenue - Second Av...,297
2,3,11215,Brooklyn,Seventh Ave (Union St - Ninth Street),141
3,4,11226,Brooklyn,Flatbush Ave (Fifth Ave - Grand Army Plaza),105
4,5,10036,Manhattan,Fifth Avenue (42nd Street - 49th Street),583
5,6,11238,Brooklyn,Flatbush Ave (Fifth Ave - Grand Army Plaza),105
6,7,11218,Brooklyn,Fifth Ave (Union St - Ninth Street),104
7,8,11225,Brooklyn,Flatbush Ave (Fifth Ave - Grand Army Plaza),105
8,9,11211,Brooklyn,Bedford Ave (Grand St - North Eighth Street),250
9,10,11209,Brooklyn,86th Street (4th Ave - Fort Hamilton Pkwy),90


In [15]:
rent_df.to_csv(PROCESSED_DIR / "commercial_rent_top10_clean.csv", index=False)

# Summary
1. Cleaned and standardized the raw source tables, including ZIP identifiers, borough names, and store-location fields.
2. Combined Whole Foods and Wegmans store lists into a unified competitor dataset, added brand, and created a standardized full_address field.
3. Geocoded competitor addresses, manually corrected known problem locations (including Union Square and Columbus Circle), and removed non-standard locations such as Industry City.
4. Joined the cleaned datasets into consistent ZIP-level spatial tables and exported the processed competitor files for use in later overlay analysis.